# 1.1 从 Unicode 到 UTF-8 字节

文本进入分词器前，要经过「字符 → Unicode 码点 → UTF-8 字节」这条链路。byte-level BPE 在最后的字节序列上学习合并规则。


In [7]:
# ord：字符 -> Unicode 码点；chr：Unicode 码点 -> 字符。
print(f"'A' 的 Unicode 码点：{ord('A')}")                  # -> 'A' 的 Unicode 码点：65
print(f"'你' 的 Unicode 码点：{ord('你')}")                # -> '你' 的 Unicode 码点：20320
print(f"'你' 的码点十六进制：{hex(ord('你'))}")            # -> '你' 的码点十六进制：0x4f60
print(f"码点 65 对应字符：{chr(65)}，码点 20320 对应字符：{chr(20320)}")  # -> 码点 65 对应字符：A，码点 20320 对应字符：你

'A' 的 Unicode 码点：65
'你' 的 Unicode 码点：20320
'你' 的码点十六进制：0x4f60
码点 65 对应字符：A，码点 20320 对应字符：你


In [8]:
text = "A你😀"
encoded = text.encode('utf-8')  # str → UTF‑8 字节序列
c1, c2, c3 = text[0], text[1], text[2]

print(f"'A' 的 UTF‑8 字节：{c1.encode('utf-8')}，字节十进制列表：{list(c1.encode('utf-8'))}")
print(f"'你' 的 UTF‑8 字节：{c2.encode('utf-8')}，字节十进制列表：{list(c2.encode('utf-8'))}")
print(f"'😀' 的 UTF‑8 字节：{c3.encode('utf-8')}，字节十进制列表：{list(c3.encode('utf-8'))}")
print(f"\n字符拼接后整体 UTF‑8：{encoded}，整体字节十进制列表：{list(encoded)}")
print(f"字节序列解码回原始字符串：{encoded.decode('utf-8')}")  # bytes → str

# bytes 遍历得到 0~255 范围的整数，这就是 Byte‑level BPE（BBPE）的输入形式


'A' 的 UTF‑8 字节：b'A'，字节十进制列表：[65]
'你' 的 UTF‑8 字节：b'\xe4\xbd\xa0'，字节十进制列表：[228, 189, 160]
'😀' 的 UTF‑8 字节：b'\xf0\x9f\x98\x80'，字节十进制列表：[240, 159, 152, 128]

字符拼接后整体 UTF‑8：b'A\xe4\xbd\xa0\xf0\x9f\x98\x80'，整体字节十进制列表：[65, 228, 189, 160, 240, 159, 152, 128]
字节序列解码回原始字符串：A你😀


In [9]:
# 问题 1
# (a) 比较同一输入在 UTF-8、UTF-16 和 UTF-32 下的字节长度。
test_string = "hello! 你是我大爷"
for name in ("utf-8", "utf-16-le", "utf-32-le"):
    print(name, len(test_string.encode(name)))
# -> utf-8 22 = 7 * 1 + 5 * 3
# -> utf-16-le 24 = 12 * 2
# -> utf-32-le 48 = 12 * 4

# 1. UTF-8 兼容 ASCII 字符，这些常见的字符只占 1 个字节, 比 UTF-16 和 UTF-32 更省空间；
# 2. UTF-8 在真实世界中的使用频率远高于另外两种，更贴合实际

utf-8 22
utf-16-le 24
utf-32-le 48


In [11]:
# (b) 该函数在解码多字节字符时会出错：逐字节解码会把多字节字符拆开，每个字节单独解码，这肯定是没道理的。
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join(bytes([b]).decode("utf-8") for b in bytestring) # (x for x in iter) 为显式生成器，当作为函数唯一参数时，可以简写成 `join(x for x in ...)`

for b in b'\xe4\xbd\xa0':  # 遍历 bytes 的每个字节产出 int：228,189,160
    print(b)

s = "-".join(["a","b"]) # 用 '-' 将列表里的所有字符拼接起来，得到 'a-b'
print(f"join 产出的结果：{s}")



228
189
160
join 产出的结果：a-b


In [12]:
print(decode_utf8_bytes_to_str_wrong("hello".encode("utf-8")))
# -> hello
try:
    decode_utf8_bytes_to_str_wrong("你好".encode("utf-8"))
except UnicodeDecodeError as e:
    print(type(e).__name__)
# -> UnicodeDecodeError

# 内置 decode 方法的实现思路：读当前首字节 → 判定字节长度，聚合连续多字节算出 Unicode 码点 → `chr()` 转字符，继续遍历后续字节，重复该过程。

hello
UnicodeDecodeError


In [13]:
# (c) 无法被解码的示例: b'\xe6\x88'
# 这一串不符合 utf-8 编码规则: 1110 开头的一定是三字节编码
bs = b'\xe6\x88'
print(' '.join(f"{b:08b}" for b in bs))
try:
    bs.decode("utf-8")
except UnicodeDecodeError as e:
    print(type(e).__name__)
# -> UnicodeDecodeError

11100110 10001000
UnicodeDecodeError
